### 库

In [3]:
# -*- coding: utf-8 -*-

import pandas as pd
import os
from tqdm import tqdm
import numpy as np
import scipy.sparse as sps

from RecSys_Course_AT_PoliMi.Data_manager.split_functions.split_train_validation_random_holdout import split_train_in_two_percentage_global_sample
from RecSys_Course_AT_PoliMi.Evaluation.Evaluator import EvaluatorHoldout
from RecSys_Course_AT_PoliMi.Recommenders.Recommender_import_list import *
import lightgbm as lgb
from sklearn.model_selection import KFold
from Recommenders.BaseRecommender import BaseRecommender

# 数据文件路径
DATA_FOLDER = 'dataset'
DATA_TRAIN_PATH = os.path.join(DATA_FOLDER, 'data_train.csv')
DATA_TARGET_USERS_PATH = os.path.join(DATA_FOLDER, 'data_target_users_test.csv')
OUTPUT_FOLDER = 'temp_output'
MODEL_FOLDER = 'model_output'
SUBMISSION_FOLDER = 'temp_output' # 提交文件的保存目录

# 随机种子，用于确保实验结果的可复现性
RANDOM_SEED = 1234

# 本地验证时，训练集所占的百分比
TRAIN_PERCENTAGE = 0.80

# 评估时使用的推荐列表长度 (cutoff)
EVALUATION_CUTOFF = 20

# 设置全局随机种子
np.random.seed(RANDOM_SEED)

best_ials_params = {
    "num_factors": 58,
    "epochs": 20,
    "confidence_scaling": "log",
    "alpha": 49.99999999999999,
    "epsilon": 5.585081217734329,
    "reg": 0.0007759360926311159
}

best_slim_params = {
    "topK": 1000,
    "l1_ratio": 0.029739176029882,
    "alpha": 0.001
}

print("项目配置加载完成.")


C:\Users\IceCould\.conda\envs\RecSysFramework\lib\site-packages\lightfm\_lightfm_fast.py:9: UserWarning: LightFM was compiled without OpenMP support. Only a single thread will be used.
  warnings.warn(


Tensorflow is not available
项目配置加载完成.


### 辅助函数

In [4]:
def load_and_preprocess_data(file_path: str) -> sps.csr_matrix:
    """
    加载CSV数据文件，并将其转换为CSR格式的稀疏矩阵.
    """
    print("--- 正在加载和预处理数据... ---")
    df_train = pd.read_csv(file_path, dtype={'row': int, 'col': int})

    n_users = df_train['row'].max() + 1
    n_items = df_train['col'].max() + 1

    urm_all = sps.coo_matrix(
        ([1.0] * len(df_train['row']), (df_train['row'], df_train['col'])),
        shape=(n_users, n_items),
        dtype=float
    ).tocsr()
    print(f"数据加载完成. URM 维度: {urm_all.shape}")
    return urm_all

def load_best_model(recommender_class, urm_train, file_name, modelfile_path):
    """
    加载一个由超参数搜索脚本保存的最佳模型。
    """
    file_path = os.path.join(modelfile_path, file_name)

    print(f"--- 正在加载预训练模型: {file_name} ---")

    # 检查模型文件是否存在
    if not os.path.exists(file_path + ".zip"):
        print(f">>> 警告: 模型文件 '{file_path}.zip' 不存在!")
        print(">>> 请确保超参数优化已完成，并且文件名正确。")
        return None

    # 1. 初始化一个“空”的模型对象
    recommender_instance = recommender_class(urm_train)

    # 2. 调用 .load_model() 方法加载数据
    recommender_instance.load_model(folder_path=modelfile_path, file_name=file_name)

    print("模型加载成功！\n")
    return recommender_instance

# --- 定义一个通用的混合推荐器类 ---
class HybridScoreRecommender(BaseRecommender):
    """
    一个通用的混合推荐器，通过加权平均多个模型的分数来进行推荐。
    """
    def __init__(self, urm_train, recommenders, weights):
        super(HybridScoreRecommender, self).__init__(urm_train)
        self.recommenders = recommenders
        self.weights = weights

    def _compute_item_score(self, user_id_array, items_to_compute=None):
        # 初始化一个零分矩阵
        final_scores = np.zeros((len(user_id_array), self.n_items), dtype=np.float32)

        for i, recommender in enumerate(self.recommenders):
            # 获取单个模型的分数
            scores = recommender._compute_item_score(user_id_array, items_to_compute)

            # 归一化分数
            scores = self._normalize_scores(scores)

            # 加权求和
            final_scores += self.weights[i] * scores

        return final_scores

    def _normalize_scores(self, scores):
        """对分数矩阵的每一行进行 Min-Max 归一化"""
        # 检查是否是稀疏矩阵，如果是，则转换为稠密数组
        if sps.issparse(scores):
            scores = scores.toarray()

        max_val = scores.max(axis=1, keepdims=True)
        min_val = scores.min(axis=1, keepdims=True)

        # 避免除以零
        denominator = max_val - min_val
        denominator[denominator == 0] = 1.0

        return (scores - min_val) / denominator

print("模型融合工具已准备就绪。")

模型融合工具已准备就绪。


### 生成数据集

In [5]:
# 加载数据
urm_all = load_and_preprocess_data(DATA_TRAIN_PATH)

# 分割数据用于本地验证
print("\n--- 正在分割数据用于本地验证... ---")
URM_train, URM_validation = split_train_in_two_percentage_global_sample(
    urm_all,
    train_percentage=TRAIN_PERCENTAGE
)
print("数据分割完成.")
print(f"训练集维度: {URM_train.shape}, 验证集维度: {URM_validation.shape}\n")


# 初始化评估器
evaluator_validation = EvaluatorHoldout(URM_validation, cutoff_list=[EVALUATION_CUTOFF])
print("评估器初始化完成.")

--- 正在加载和预处理数据... ---
数据加载完成. URM 维度: (27095, 6969)

--- 正在分割数据用于本地验证... ---
数据分割完成.
训练集维度: (27095, 6969), 验证集维度: (27095, 6969)

EvaluatorHoldout: Ignoring 37 ( 0.1%) Users that have less than 1 test interactions
评估器初始化完成.


### 确定权重

In [ ]:
# --- 配置区域 ---
FOLD_MODEL_FOLDER = "k_fold_models64e"
EVALUATION_CUTOFF = 20

# 定义要搜索的 Alpha 范围
alpha_values = np.arange(0.65, 0.75, 0.01)
alpha_results = {alpha: [] for alpha in alpha_values}

print(f"--- 开始 K-Fold ({len(alpha_values)} Alphas) 交叉验证搜索最佳权重 ---")

kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
urm_train_coo = URM_train.tocoo()

# --- 外层循环：遍历 Folds ---
for fold_index, (train_index, val_index) in enumerate(kf.split(urm_train_coo.row)):
    print(f"\n>>> 正在处理 Fold {fold_index + 1}/5 ...")

    # 1. 【关键修复】构建当前 Fold 的训练集矩阵
    # 这是模型在训练时真正见过的矩阵，不包含验证集数据
    urm_train_fold = sps.csr_matrix(
        (urm_train_coo.data[train_index], (urm_train_coo.row[train_index], urm_train_coo.col[train_index])),
        shape=URM_train.shape
    )

    # 2. 构建当前 Fold 的验证集矩阵
    urm_val_fold = sps.csr_matrix(
        (urm_train_coo.data[val_index], (urm_train_coo.row[val_index], urm_train_coo.col[val_index])),
        shape=URM_train.shape
    )

    # 3. 初始化评估器
    # 注意：exclude_seen_on 参数通常需要设为 urm_train_fold，告诉评估器哪些是训练数据
    evaluator_fold = EvaluatorHoldout(urm_val_fold, cutoff_list=[EVALUATION_CUTOFF])

    # 4. 加载模型
    slim_filename = f"SLIM_fold_{fold_index + 1}"
    ials_filename = f"IALS_fold_{fold_index + 1}"

    try:
        # 注意：这里 load_best_model 传入 urm_train_fold 是为了确保模型内部引用的 URM 是正确的
        recommender_slim_fold = load_best_model(
            recommender_class=SLIMElasticNetRecommender,
            urm_train=urm_train_fold, # <--- 修改这里：传入当前折的训练数据
            file_name=slim_filename,
            modelfile_path=FOLD_MODEL_FOLDER
        )

        recommender_ials_fold = load_best_model(
            recommender_class=IALSRecommender,
            urm_train=urm_train_fold, # <--- 修改这里：传入当前折的训练数据
            file_name=ials_filename,
            modelfile_path=FOLD_MODEL_FOLDER
        )
    except FileNotFoundError:
        print(f"    错误：找不到 Fold {fold_index + 1} 的模型文件，跳过。")
        continue

    recommender_list_fold = [recommender_slim_fold, recommender_ials_fold]

    # --- 内层循环：遍历 Alpha ---
    print(f"    正在评估 {len(alpha_values)} 个 Alpha 值...")

    for alpha in tqdm(alpha_values, desc=f"Fold {fold_index+1} Alpha Search"):
        weights = [alpha, 1 - alpha]

        # 这里必须传入 urm_train_fold，而不是全局 URM_train
        # 这样 HybridRecommender 才知道只能过滤掉 train_fold 里的物品，而保留 val_fold 里的物品
        hybrid_recommender = HybridScoreRecommender(urm_train_fold, recommender_list_fold, weights)

        results_df, _ = evaluator_fold.evaluateRecommender(hybrid_recommender)
        score = results_df.loc[EVALUATION_CUTOFF].get('RECALL', 0.0)

        # 简单的 debug 打印，确认分数正常
        print(f"Alpha {alpha:.2f} -> Score: {score}")

        alpha_results[alpha].append(score)

# --- 最终结果 ---
print("\n" + "="*40)
print("K-FOLD 结果汇总")
best_avg_score = 0
best_avg_alpha = 0

for alpha in alpha_values:
    scores = alpha_results[alpha]
    if not scores: continue
    avg_score = sum(scores) / len(scores)

    print(f"Alpha {alpha:.2f}: Avg Recall = {avg_score:.5f}")
    if avg_score > best_avg_score:
        best_avg_score = avg_score
        best_avg_alpha = alpha

print(f"\n最佳 Alpha: {best_avg_alpha:.2f}")
print(f"最佳 Recall: {best_avg_score:.5f}")

In [ ]:
# --- 1. 配置最佳权重和模型信息 ---
BEST_ALPHA = 0.25 # !!! 请确保将此值替换为您在网格搜索中找到的最佳 alpha !!!

MODEL_FOLDER = 'model_output' # 模型文件所在的文件夹

# --- 2. 加载在 *全量数据* 上训练好的模型 ---
print("\n--- 正在加载在全量数据上预训练的最终模型... ---")

# 加载 SLIMElasticNetRecommender
final_recommender_slim = load_best_model(
    recommender_class=SLIMElasticNetRecommender,
    urm_train=urm_all, # 使用 urm_all 初始化
    file_name="5-1final_model_SLIMElasticNetRecommender-better",
    modelfile_path=MODEL_FOLDER
)

# 加载 b
final_recommender_b = load_best_model(
    recommender_class=IALSRecommender,
    urm_train=urm_all, # 使用 urm_all 初始化
    file_name="5-2final_model_IALSRecommender",
    modelfile_path=MODEL_FOLDER
)


# --- 3. 创建最终的混合推荐器并生成提交 ---
if final_recommender_slim and final_recommender_b:

    final_recommender_list = [final_recommender_slim, final_recommender_b]
    final_weights = [BEST_ALPHA, 1 - BEST_ALPHA]

    final_hybrid_recommender = HybridScoreRecommender(urm_all, final_recommender_list, final_weights)

    # --- 开始生成推荐 ---
    df_target_users = pd.read_csv(DATA_TARGET_USERS_PATH)
    target_user_ids = df_target_users['user_id'].values

    submission = []
    print(f"--- 正在为 {len(target_user_ids)} 名目标用户生成最终融合推荐... ---")
    for user_id in tqdm(target_user_ids):
        recommended_items = final_hybrid_recommender.recommend(
            user_id,
            cutoff=EVALUATION_CUTOFF,
            remove_seen_flag=True
        )
        submission.append((user_id, ' '.join(map(str, recommended_items))))

    # --- 保存提交文件 ---
    submission_filename = f"submission_FINAL_Hybrid_SLIM_IALS_alpha_{BEST_ALPHA:.2f}.csv"
    SUBMISSION_PATH = os.path.join(SUBMISSION_FOLDER, submission_filename) # SUBMISSION_FOLDER 在之前已定义

    df_submission = pd.DataFrame(submission, columns=['user_id', 'item_list'])
    df_submission.to_csv(SUBMISSION_PATH, index=False)

    print(f"\n--- 最终提交文件已成功生成! ---")
    print(f"文件保存在: {SUBMISSION_PATH}")
else:
    print("\n>>> 错误：一个或多个最终模型加载失败，无法生成提交文件。")

### 000

In [6]:
# 设定每个模型提取的候选数量 (越大越好，建议 100-200)
CANDIDATE_CUTOFF = 200

def get_candiates_dataframe(recommender, target_users):
    """
    使用模型对 target_users 进行预测，并返回格式化的 DataFrame。
    带有详细的进度打印。
    """
    user_list = []
    item_list = []
    score_list = []

    # 这里设置 batch_size 是为了防止内存爆掉，同时也方便显示进度条
    batch_size = 1000

    # 分批次处理
    for start_idx in tqdm(range(0, len(target_users), batch_size), desc=f"正在提取 {recommender.RECOMMENDER_NAME} 的分数"):
        end_idx = min(start_idx + batch_size, len(target_users))
        batch_users = target_users[start_idx:end_idx]

        # 1. 推荐 (返回 items)
        # remove_seen_flag=True 会自动利用模型内部存储的 URM_train 过滤已读
        recommendations = recommender.recommend(
            batch_users,
            cutoff=CANDIDATE_CUTOFF,
            remove_seen_flag=True
        )

        # 2. 获取分数
        # 注意：这里假设 recommender 有 _compute_item_score 方法
        # 如果没有，可能需要检查你的框架 recommend 是否直接返回 (items, scores)
        all_scores_batch = recommender._compute_item_score(batch_users)

        for i, user_id in enumerate(batch_users):
            rec_items = recommendations[i]
            # 从密集矩阵中只提取推荐物品的分数
            rec_scores = all_scores_batch[i, rec_items]

            user_list.extend([user_id] * len(rec_items))
            item_list.extend(rec_items)
            score_list.extend(rec_scores)

    df = pd.DataFrame({
        "user_id": user_list,
        "item_id": item_list,
        "score": score_list
    })

    return df

In [8]:
import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.model_selection import KFold
from tqdm import tqdm
import os

# ==========================================
# 0. 配置与路径
# ==========================================
MODEL_FOLDER = 'k_fold_models_64e' # 你的模型文件夹
if not os.path.exists(MODEL_FOLDER): os.makedirs(MODEL_FOLDER)

# 5-Fold 切分器
n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_SEED)

# 获取 80% 训练数据的交互索引 (用于切分)
urm_80_coo = URM_train.tocoo()
interactions_indices = np.arange(urm_80_coo.nnz)

# 准备 20% Holdout 的目标用户
target_users_holdout = np.unique(URM_validation.indices)

print(f"数据准备: 80% 训练集交互数 {urm_80_coo.nnz}, 20% Holdout 用户数 {len(target_users_holdout)}")

# 容器
oof_dfs = []
holdout_preds = []

# ==========================================
# 1. 循环 5 个 Fold
# ==========================================
fold_id = 0

for train_idx, val_idx in kf.split(interactions_indices):
    fold_id += 1
    print(f"\n>>> 处理 Fold {fold_id} / {n_splits} ...")

    # --- A. 构建当前 Fold 的训练矩阵 (64% 数据) ---
    train_mask = np.zeros(urm_80_coo.nnz, dtype=bool)
    train_mask[train_idx] = True

    URM_train_fold = sp.csr_matrix(
        (urm_80_coo.data[train_mask], (urm_80_coo.row[train_mask], urm_80_coo.col[train_mask])),
        shape=URM_train.shape
    )

    # 确定 OOF (16%) 的目标用户
    val_mask = np.zeros(urm_80_coo.nnz, dtype=bool)
    val_mask[val_idx] = True
    target_users_oof = np.unique(urm_80_coo.row[val_mask])

    # [预留的读取注释代码]
    # 如果下次不想重训，直接取消下面两行的注释，并注释掉上面的 fit 和 save_model
    print(f"加载 UserKNN (Fold {fold_id})...")
    rec_user = UserKNNCFRecommender(URM_train_fold)
    rec_user.load_model(MODEL_FOLDER, user_knn_filename)

    # ====================================================
    # --- C. 加载已有的 SLIM 和 IALS ---
    # ====================================================
    print(f"加载已有的 SLIM 和 IALS (Fold {fold_id})...")

    # 1. 加载 IALS (文件名格式: IALS_fold_1)
    rec_ials = IALSRecommender(URM_train_fold)
    rec_ials.load_model(MODEL_FOLDER, f"IALS_fold_{fold_id}")

    # 2. 加载 SLIM (假设文件名格式: SLIM_fold_1，如果不同请修改这里!)
    # 如果你的 SLIM 名字不同，请修改 file_name 参数
    try:
        rec_slim = SLIMElasticNetRecommender(URM_train_fold)
        rec_slim.load_model(MODEL_FOLDER, f"SLIM_fold_{fold_id}")
    except:
        print("警告: 没找到 SLIM_fold_X，尝试使用其他命名...")
        # 这里写你实际的 SLIM 文件名逻辑，例如:
        rec_slim.load_model(MODEL_FOLDER, f"SLIMElasticNetRecommender_fold_{fold_id}")

    # ====================================================
    # --- D. 生成预测 (OOF + Holdout) ---
    # ====================================================
    print("生成 OOF (16%) 预测...")
    df_slim_oof = get_candiates_dataframe(rec_slim, target_users_oof)
    df_ials_oof = get_candiates_dataframe(rec_ials, target_users_oof)
    df_user_oof = get_candiates_dataframe(rec_user, target_users_oof)

    # 合并 OOF
    df_fold_oof = pd.merge(df_slim_oof, df_ials_oof, on=["user_id", "item_id"], how="outer")
    df_fold_oof = pd.merge(df_fold_oof, df_user_oof, on=["user_id", "item_id"], how="outer")
    oof_dfs.append(df_fold_oof)

    print("生成 Holdout (20%) 预测...")
    # 注意：这里我们分别保存，为了最后做 Bagging
    holdout_preds.append({
        "slim": get_candiates_dataframe(rec_slim, target_users_holdout),
        "ials": get_candiates_dataframe(rec_ials, target_users_holdout),
        "user": get_candiates_dataframe(rec_user, target_users_holdout)
    })

    # 内存清理
    del rec_slim, rec_ials, rec_user, URM_train_fold
    import gc; gc.collect()

# ==========================================
# 2. 数据整理 (为下一步验证做准备)
# ==========================================
print("\n>>> 正在整合数据...")

# 1. 整合 OOF
df_oof_all = pd.concat(oof_dfs, ignore_index=True)
df_oof_all.columns = ["user_id", "item_id", "slim_score", "ials_score", "user_score"]
df_oof_all.fillna(0.0, inplace=True)

# 2. 整合 Holdout (Bagging Average)
print("计算 Holdout 的 Bagging 平均分...")
def bag_models(preds_list, col_name):
    df_cat = pd.concat(preds_list)
    return df_cat.groupby(["user_id", "item_id"])["score"].mean().reset_index().rename(columns={"score": col_name})

df_holdout_slim = bag_models([p['slim'] for p in holdout_preds], "slim_score")
df_holdout_ials = bag_models([p['ials'] for p in holdout_preds], "ials_score")
df_holdout_user = bag_models([p['user'] for p in holdout_preds], "user_score")

df_holdout_all = pd.merge(df_holdout_slim, df_holdout_ials, on=["user_id", "item_id"], how="outer")
df_holdout_all = pd.merge(df_holdout_all, df_holdout_user, on=["user_id", "item_id"], how="outer")
df_holdout_all.fillna(0.0, inplace=True)

# 3. 打标签 (Ground Truth)
print("正在打标签 (Labeling)...")
# OOF 标签来自 URM_train (80%)
gt_80 = set(zip(urm_80_coo.row, urm_80_coo.col))
df_oof_all['label'] = [1 if (u,i) in gt_80 else 0 for u,i in zip(df_oof_all['user_id'], df_oof_all['item_id'])]

# Holdout 标签来自 URM_validation (20%)
urm_20_coo = URM_validation.tocoo()
gt_20 = set(zip(urm_20_coo.row, urm_20_coo.col))
df_holdout_all['label'] = [1 if (u,i) in gt_20 else 0 for u,i in zip(df_holdout_all['user_id'], df_holdout_all['item_id'])]

print("✅ 完成！现在你可以运行之前的【验证核心逻辑 (Search vs Verify)】代码了。")
# 你现在可以直接使用 df_oof_all 和 df_holdout_all 变量

数据准备: 80% 训练集交互数 2434446, 20% Holdout 用户数 6960

>>> 处理 Fold 1 / 5 ...
加载 UserKNN (Fold 1)...
UserKNNCFRecommender: Loading model from file 'k_fold_models_64eUserKNNCF_fold_1'
UserKNNCFRecommender: Loading complete
加载已有的 SLIM 和 IALS (Fold 1)...
IALSRecommender: Loading model from file 'k_fold_models_64eIALS_fold_1'
IALSRecommender: Loading complete
SLIMElasticNetRecommender: Loading model from file 'k_fold_models_64eSLIM_fold_1'


正在提取 SLIMElasticNetRecommender 的分数:   0%|          | 0/27 [00:00<?, ?it/s]

SLIMElasticNetRecommender: Loading complete
生成 OOF (16%) 预测...


正在提取 SLIMElasticNetRecommender 的分数:   0%|          | 0/7 [00:00<?, ?it/s]

生成 Holdout (20%) 预测...


...

SLIMElasticNetRecommender: Loading model from file 'k_fold_models_64eSLIM_fold_5'
SLIMElasticNetRecommender: Loading complete
生成 OOF (16%) 预测...


正在提取 SLIMElasticNetRecommender 的分数:   0%|          | 0/7 [00:00<?, ?it/s]

生成 Holdout (20%) 预测...


正在提取 UserKNNCFRecommender 的分数: 100%|██████████| 7/7 [00:01<00:00,  3.97it/s]



>>> 正在整合数据...
计算 Holdout 的 Bagging 平均分...
正在打标签 (Labeling)...
✅ 完成！现在你可以运行之前的【验证核心逻辑 (Search vs Verify)】代码了。


In [9]:
import numpy as np
import pandas as pd

# 假设 URM_validation 是你的验证集稀疏矩阵 (csr_matrix)
# 如果是 OOF 阶段，这里应该用 URM_train (80%那个)
# 如果是 Holdout 阶段，这里应该用 URM_validation (20%那个)

def get_user_truth_counts(URM):
    """
    预计算每个用户的真实交互数量 (分母)
    """
    # np.diff(indptr) 是 CSR 矩阵快速获取每行非零元素个数的标准方法
    user_len = np.diff(URM.indptr)

    # 转成 Series，方便后续按 user_id 索引对齐
    truth_counts = pd.Series(user_len, index=np.arange(URM.shape[0]))

    # 过滤掉那些在验证集里完全没有交互的用户（防止除以0）
    truth_counts = truth_counts[truth_counts > 0]
    return truth_counts

# --- 请根据当前环节生成对应的分母 ---
# 1. 针对 OOF 调参 (分母来自 80% 训练集)
gt_counts_oof = get_user_truth_counts(URM_train)

# 2. 针对 Holdout 验证 (分母来自 20% 验证集)
gt_counts_holdout = get_user_truth_counts(URM_validation)

In [10]:
def calculate_recall_vectorized(df_scored, gt_counts, k=20):
    """
    基于 DataFrame 的向量化 Recall 计算
    参数:
        df_scored: 包含 ['user_id', 'final_score', 'label'] 的 DataFrame
        gt_counts: 这是一个 Series，索引是 user_id，值是该用户的真实交互总数
        k: 截断值 (默认20)
    """
    # 1. 排序：按用户分组，分数降序
    # (这一步是耗时大头，但 Pandas 底层优化过，比手写快)
    df_sorted = df_scored.sort_values(["user_id", "final_score"], ascending=[True, False])

    # 2. 截断：每个用户只取前 K 个
    df_topk = df_sorted.groupby("user_id").head(k)

    # 3. 聚合：计算每个用户预测对了几个 (Hits)
    # 因为 label 是 0 或 1，sum() 就是命中数
    user_hits = df_topk.groupby("user_id")['label'].sum()

    # 4. 计算：分子(Hits) / 分母(Truth Counts)
    # 使用索引对齐，只计算两者交集用户
    common_users = user_hits.index.intersection(gt_counts.index)

    if len(common_users) == 0:
        return 0.0

    recall_series = user_hits.loc[common_users] / gt_counts.loc[common_users]

    # 5. 返回平均值
    return recall_series.mean()

In [12]:
from sklearn.preprocessing import MinMaxScaler

# ==========================================
# 0. 准备 Recall 计算所需的“分母”
# ==========================================
print("正在准备 Recall 计算工具...")

# 1. 计算 OOF 数据集中，每个用户的真实交互数 (分母)
# 来源：URM_train (80% 数据)
# 注意：我们要找的是那些出现在 OOF 目标列表里的用户
user_gt_count_oof = pd.Series(np.diff(URM_train.indptr), index=np.arange(URM_train.shape[0]))
# 过滤掉交互为0的用户(防止除以0)，虽然一般没有
user_gt_count_oof = user_gt_count_oof[user_gt_count_oof > 0]

# 2. 计算 Holdout 数据集中，每个用户的真实交互数 (分母)
# 来源：URM_validation (20% 数据)
user_gt_count_holdout = pd.Series(np.diff(URM_validation.indptr), index=np.arange(URM_validation.shape[0]))
user_gt_count_holdout = user_gt_count_holdout[user_gt_count_holdout > 0]

# ==========================================
# 1. 定义真正的 Recall 计算函数
# ==========================================
def calculate_true_recall(df_scored, gt_count_series):
    """
    df_scored: 包含 user_id, final_score, label 的 DataFrame
    gt_count_series: 索引为 user_id，值为该用户真实交互数的 Series
    """
    # 1. 排序并取 Top 20
    # 技巧：使用 nlargest 或 sort_values + head
    # 这里假设 df_scored 已经是所有候选集了

    # 排序：按用户分组，分数降序
    df_sorted = df_scored.sort_values(["user_id", "final_score"], ascending=[True, False])

    # 取每个用户的 Top 20
    df_top20 = df_sorted.groupby("user_id").head(20)

    # 2. 计算分子：每个用户预测对了几个 (Hits)
    user_hits = df_top20.groupby("user_id")['label'].sum()

    # 3. 计算 Recall：分子 / 分母
    # 注意：我们要确保只计算那些在 gt_count_series 里存在的用户
    # 两个 Series 对齐索引
    common_users = user_hits.index.intersection(gt_count_series.index)

    recall_series = user_hits.loc[common_users] / gt_count_series.loc[common_users]

    # 4. 返回平均值
    return recall_series.mean()

# 1. 准备分母 (只用于 Holdout)
# OOF 阶段我们就用 hits，因为算 OOF 的精确分母太麻烦且没必要
gt_counts_holdout = get_user_truth_counts(URM_validation) # 20% 那部分的分母

print("\n>>> 阶段 1: 在 OOF (16%) 上搜索最佳权重 <<<")
print("    (注: OOF 使用 Hits 作为指标，因为 Recall 分母会被稀释，不影响排序)")

best_w = (0, 0, 0)
best_score_oof = 0 # 这里存 Hits

# 预先 MinMax
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
for col in ["slim_score", "ials_score", "user_score"]:
    # 也就是对 df_oof_all 和 df_holdout_all 都做处理
    df_oof_all[f"{col}_mm"] = scaler.fit_transform(df_oof_all[col].values.reshape(-1, 1))
    df_holdout_all[f"{col}_mm"] = scaler.fit_transform(df_holdout_all[col].values.reshape(-1, 1))

# --- 搜索循环 ---
for w_slim in np.arange(0.6, 0.95, 0.05):
    for w_user in np.arange(0.0, 0.01, 0.05):
        w_ials = 1.0 - w_slim - w_user
        if w_user < 0 or w_user > 0.4: continue

        # 计算 OOF 分数
        df_oof_all["final_score"] = (w_slim * df_oof_all["slim_score_mm"] +
                                     w_ials * df_oof_all["ials_score_mm"] +
                                     w_user * df_oof_all["user_score_mm"])

        # 【关键】OOF 阶段只看总命中数 (Hits)，这足以判断权重好坏
        # 排序取 Top 20
        # 向量化计算 Hits: 直接利用 Label 求和
        # (这里为了速度，我们假设 df_oof_all 已经是所有候选集了，直接 sort + head)
        current_hits = df_oof_all.sort_values(["user_id", "final_score"], ascending=[False, False]) \
                                 .groupby("user_id").head(20)['label'].sum()

        if current_hits > best_score_oof:
            best_score_oof = current_hits
            best_w = (w_slim, w_ials, w_user)
            print(f"OOF 新高 Hits: {int(current_hits)} | Weights: {best_w}")

print(f"\n✅ OOF 锁定的最佳权重: {best_w}")


print(f"\n>>> 阶段 2: 在 Holdout (20%) 上验证真值 <<<")
print("    (注: 这里计算的是真正的 Recall@20)")

# 应用最佳权重到 Holdout
df_holdout_all["final_score"] = (best_w[0] * df_holdout_all["slim_score_mm"] +
                                 best_w[1] * df_holdout_all["ials_score_mm"] +
                                 best_w[2] * df_holdout_all["user_score_mm"])

# 计算真正的 Recall
# 使用之前的 calculate_recall_vectorized 函数
real_recall = calculate_recall_vectorized(df_holdout_all, gt_counts_holdout, k=20)

print(f"🏆 最终 Holdout Recall@20: {real_recall:.5f}")

if real_recall > 0.29:
    print(">>> 恭喜！你的策略成功突破 0.29，可以进行全量 Bagging 提交了！")
else:
    print(f">>> 当前分数为 {real_recall:.5f}，请检查 UserKNN 是否正常输出了有效分数。")

正在准备 Recall 计算工具...

>>> 阶段 1: 在 OOF (16%) 上搜索最佳权重 <<<
    (注: OOF 使用 Hits 作为指标，因为 Recall 分母会被稀释，不影响排序)
OOF 新高 Hits: 100562 | Weights: (0.6, 0.4, 0.0)
OOF 新高 Hits: 102068 | Weights: (0.65, 0.35, 0.0)
OOF 新高 Hits: 103536 | Weights: (0.7000000000000001, 0.29999999999999993, 0.0)
OOF 新高 Hits: 104826 | Weights: (0.7500000000000001, 0.2499999999999999, 0.0)
OOF 新高 Hits: 106027 | Weights: (0.8000000000000002, 0.19999999999999984, 0.0)
OOF 新高 Hits: 107131 | Weights: (0.8500000000000002, 0.1499999999999998, 0.0)
OOF 新高 Hits: 108115 | Weights: (0.9000000000000002, 0.09999999999999976, 0.0)

✅ OOF 锁定的最佳权重: (0.9000000000000002, 0.09999999999999976, 0.0)

>>> 阶段 2: 在 Holdout (20%) 上验证真值 <<<
    (注: 这里计算的是真正的 Recall@20)
🏆 最终 Holdout Recall@20: 0.19865
>>> 当前分数为 0.19865，请检查 UserKNN 是否正常输出了有效分数。


# 1

In [19]:
import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.model_selection import KFold
from tqdm import tqdm

# 假设 URM_train 是你的全量交互矩阵 (csr_matrix)
n_splits = 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_SEED)

# 获取所有交互的坐标 (row=user, col=item)
# 这样我们可以对交互进行切分
urm_coo = URM_train.tocoo()
interactions_indices = np.arange(urm_coo.nnz)

print(f"开始 5-Fold OOF 生成流程，总交互数: {urm_coo.nnz}")

开始 5-Fold OOF 生成流程，总交互数: 2434446


In [31]:
# 修正 OOF 生成过程（跳过训练，直接加载模型预测）

import scipy.sparse as sp
import numpy as np
import pandas as pd

oof_dfs = []
fold_id = 0

# 重新遍历 K-Fold (为了获取正确的验证集 indices)
for train_idx, val_idx in kf.split(interactions_indices):
    fold_id += 1
    print(f"\n========== Fold {fold_id} / {n_splits} ==========")

    # 1. 重建验证集矩阵 (只为了获取正确的 Target Users)
    # 这一步很快，不耗时
    val_mask = np.zeros(urm_coo.nnz, dtype=bool)
    val_mask[val_idx] = True

    # 必须用原始 shape
    URM_val_fold = sp.csr_matrix(
        (urm_coo.data[val_mask], (urm_coo.row[val_mask], urm_coo.col[val_mask])),
        shape=urm_coo.shape
    )

    # ================= 修正点 =================
    # 获取行索引（User IDs），而不是 indices（Item IDs）
    # 使用 .tocoo().row 是最安全直观的方法
    target_users_fold = np.unique(URM_val_fold.tocoo().row)
    print(f"修正后的目标用户数: {len(target_users_fold)}")
    # =========================================

    # 2. 加载已保存的模型 (跳过 fit 过程)
    print(f"加载模型 Fold {fold_id} ...")

    # 假设你之前保存了，现在直接 load
    # 注意：加载时通常需要传入 URM_train_fold 或者 urm_all 以便过滤 seen items
    # 这里我们重新构建一下 URM_train_fold 仅用于过滤
    train_mask = np.zeros(urm_coo.nnz, dtype=bool)
    train_mask[train_idx] = True
    URM_train_fold = sp.csr_matrix(
        (urm_coo.data[train_mask], (urm_coo.row[train_mask], urm_coo.col[train_mask])),
        shape=urm_coo.shape
    )

    # 加载 SLIM
    rec_slim_fold = SLIMElasticNetRecommender(URM_train_fold)
    rec_slim_fold.load_model("k_fold_models", f"SLIM_fold_{fold_id}")

    # 加载 IALS
    rec_ials_fold = IALSRecommender(URM_train_fold)
    rec_ials_fold.load_model("k_fold_models", f"IALS_fold_{fold_id}")

    # 加载 UserKNN
    rec_user_fold = UserKNNCFRecommender(URM_train_fold)
    rec_user_fold.load_model("k_fold_models", f"UserKNN_fold_{fold_id}")

    # 3. 重新生成预测 (使用正确的 target_users_fold)
    print("重新生成预测...")
    df_slim_f = get_candiates_dataframe(rec_slim_fold, target_users_fold)
    df_slim_f.rename(columns={"score": "slim_score"}, inplace=True)

    df_ials_f = get_candiates_dataframe(rec_ials_fold, target_users_fold)
    df_ials_f.rename(columns={"score": "ials_score"}, inplace=True)

    df_user_f = get_candiates_dataframe(rec_user_fold, target_users_fold)
    df_user_f.rename(columns={"score": "user_score"}, inplace=True)

    # 4. 合并与打标签 (和之前一样)
    df_fold = pd.merge(df_slim_f, df_ials_f, on=["user_id", "item_id"], how="outer")
    df_fold = pd.merge(df_fold, df_user_f, on=["user_id", "item_id"], how="outer")

    for col in ["slim_score", "ials_score", "user_score"]:
        df_fold[col].fillna(0.0, inplace=True)

    # 打标签逻辑
    val_coo = URM_val_fold.tocoo()
    # 优化速度：转成 DataFrame merge
    gt_df = pd.DataFrame({'user_id': val_coo.row, 'item_id': val_coo.col, 'label': 1})
    df_fold = pd.merge(df_fold, gt_df, on=['user_id', 'item_id'], how='left')
    df_fold['label'].fillna(0, inplace=True)

    oof_dfs.append(df_fold)

# 保存修正后的 OOF
df_oof_corrected = pd.concat(oof_dfs, ignore_index=True)
df_oof_corrected.to_csv("oof_predictions_3models_corrected.csv", index=False)


========== Fold 1 / 5 ==========
修正后的目标用户数: 26995
加载模型 Fold 1 ...
SLIMElasticNetRecommender: Loading model from file 'k_fold_modelsSLIM_fold_1'
SLIMElasticNetRecommender: Loading complete
IALSRecommender: Loading model from file 'k_fold_modelsIALS_fold_1'
IALSRecommender: Loading complete
UserKNNCFRecommender: Loading model from file 'k_fold_modelsUserKNN_fold_1'
UserKNNCFRecommender: Loading complete
重新生成预测...

========== Fold 2 / 5 ==========
修正后的目标用户数: 27005
加载模型 Fold 2 ...
SLIMElasticNetRecommender: Loading model from file 'k_fold_modelsSLIM_fold_2'
SLIMElasticNetRecommender: Loading complete
IALSRecommender: Loading model from file 'k_fold_modelsIALS_fold_2'
IALSRecommender: Loading complete
UserKNNCFRecommender: Loading model from file 'k_fold_modelsUserKNN_fold_2'
UserKNNCFRecommender: Loading complete
重新生成预测...

========== Fold 3 / 5 ==========
修正后的目标用户数: 27016
加载模型 Fold 3 ...
SLIMElasticNetRecommender: Loading model from file 'k_fold_modelsSLIM_fold_3'
SLIMElasticNetRecommend

In [22]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.preprocessing import MinMaxScaler
from tqdm import tqdm

# --- 配置：请修改这里以匹配你的 CSV 列名 ---
CSV_PATH = "oof_predictions_3models_corrected.csv"

# 假设你的CSV列名如下，如果不同请修改：
COL_USER = "user_id"
COL_ITEM = "item_id"
COL_LABEL = "label"  # 或者是 "target", "interaction"
COL_SLIM = "slim_score" # SLIM 的预测分
COL_IALS = "ials_score" # IALS 的预测分
COL_USERKNN = "user_score" # UserKNN 的预测分

# ----------------------------------------

print("--- 1. 加载 OOF 数据 ---")
df = pd.read_csv(CSV_PATH)

# 简单检查
print(f"数据量: {len(df)}")
print("列名:", df.columns.tolist())

# 确保分数列是数值型，并填充 NaN
for col in [COL_SLIM, COL_IALS, COL_USERKNN]:
    df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0.0)

# --- 2. 特征工程 (复刻之前成功的逻辑) ---
print("\n--- 2. 构建 Stacking 特征 ---")

# A. Rank 特征 (LGBM 最喜欢的)
print("计算 Rank (排名)...")
# method='min': 第1名rank是1。ascending=False: 分数越高rank越小
df['slim_rank'] = df.groupby(COL_USER)[COL_SLIM].rank(ascending=False, method='min')
df['ials_rank'] = df.groupby(COL_USER)[COL_IALS].rank(ascending=False, method='min')
df['user_rank'] = df.groupby(COL_USER)[COL_USERKNN].rank(ascending=False, method='min')

# B. MinMax 特征 (归一化)
print("计算 MinMax (归一化)...")
scaler = MinMaxScaler()
df['slim_mm'] = scaler.fit_transform(df[COL_SLIM].values.reshape(-1, 1))
df['ials_mm'] = scaler.fit_transform(df[COL_IALS].values.reshape(-1, 1))
df['user_mm'] = scaler.fit_transform(df[COL_USERKNN].values.reshape(-1, 1))

# C. 差值特征
df['diff_slim_user'] = df['slim_mm'] - df['user_mm']
df['diff_slim_ials'] = df['slim_mm'] - df['ials_mm']

# 定义最终送入模型的特征列表
feature_cols = [
    COL_SLIM, COL_IALS, COL_USERKNN,
    'slim_rank', 'ials_rank', 'user_rank',
    'slim_mm', 'ials_mm', 'user_mm',
    'diff_slim_user', 'diff_slim_ials'
]

print("特征准备完毕。")

--- 1. 加载 OOF 数据 ---
数据量: 44905412
列名: ['user_id', 'item_id', 'slim_score', 'ials_score', 'user_score', 'label']

--- 2. 构建 Stacking 特征 ---
计算 Rank (排名)...
计算 MinMax (归一化)...
特征准备完毕。


In [23]:
print("\n--- 3. 划分 Stacking 的 训练集/验证集 ---")

# 必须按用户切分，不能按行切分
all_users = df[COL_USER].unique()
np.random.shuffle(all_users)

split_idx = int(len(all_users) * 0.8)
train_users = all_users[:split_idx]
valid_users = all_users[split_idx:]

df_train = df[df[COL_USER].isin(train_users)].copy()
df_valid = df[df[COL_USER].isin(valid_users)].copy()

# LGBMRanker 要求数据必须按 Group 排序
df_train = df_train.sort_values(by=[COL_USER])
df_valid = df_valid.sort_values(by=[COL_USER])

# 获取 group 信息 (每个用户有多少个候选物品)
qids_train = df_train.groupby(COL_USER).size().values
qids_valid = df_valid.groupby(COL_USER).size().values

# 准备 X, y
X_train = df_train[feature_cols]
y_train = df_train[COL_LABEL]

X_valid = df_valid[feature_cols]
y_valid = df_valid[COL_LABEL]

print(f"Stacking 训练用户数: {len(train_users)}")
print(f"Stacking 验证用户数: {len(valid_users)}")

# --- 4. 训练模型 ---
print("\n--- 4. 开始训练 LGBMRanker ---")

ranker = lgb.LGBMRanker(
    objective="lambdarank",
    metric="ndcg",
    n_estimators=1000,      # 最大树数量
    learning_rate=0.05,
    num_leaves=31,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    importance_type='gain'  # 看看谁贡献最大
)

ranker.fit(
    X_train, y_train,
    group=qids_train,
    eval_set=[(X_valid, y_valid)],
    eval_group=[qids_valid],
    eval_at=[20],          # 关注 Top 20 的效果
)


--- 3. 划分 Stacking 的 训练集/验证集 ---
Stacking 训练用户数: 21676
Stacking 验证用户数: 5419

--- 4. 开始训练 LGBMRanker ---
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.217346 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 2805
[LightGBM] [Info] Number of data points in the train set: 35934154, number of used features: 11


LGBMRanker(colsample_bytree=0.8, importance_type='gain', learning_rate=0.05,
           metric='ndcg', n_estimators=1000, objective='lambdarank',
           random_state=42, subsample=0.8)

In [24]:
print("\n--- 5. 最终效果评估 ---")

# 1. 查看特征重要性
importance = pd.DataFrame({
    'Feature': feature_cols,
    'Gain': ranker.feature_importances_
}).sort_values('Gain', ascending=False)

print("\n[特征重要性排行榜]")
print(importance)

# 2. 计算 Recall@20 (在 Stacking 的验证集上)
# 使用模型预测
df_valid['final_score'] = ranker.predict(df_valid[feature_cols])

# 排序取 Top 20
recs = df_valid.sort_values(by=[COL_USER, 'final_score'], ascending=[True, False])
recs = recs.groupby(COL_USER).head(20)

# 计算 Recall
# 既然是 OOF CSV，里面应该包含了 label=1 的真实数据
# 我们统计命中数
hits = recs[recs[COL_LABEL] == 1].groupby(COL_USER).size()
# 统计每个用户实际有多少个正样本 (分母)
# 注意：这里的 df_valid 是候选集，可能不包含所有的正样本(如果Base Model没召回的话)
# 但为了对比 Stacking 提升，我们通常假设 candidate 里的 label=1 就是我们要找的
gt_counts = df_valid[df_valid[COL_LABEL] == 1].groupby(COL_USER).size()

# 对齐索引
all_valid_users_idx = pd.Index(valid_users)
hits = hits.reindex(all_valid_users_idx, fill_value=0)
gt_counts = gt_counts.reindex(all_valid_users_idx, fill_value=0) # 防止除以0，稍后处理

# 计算 Recall (Micro 或 Macro)
# 这里简单计算：总命中数 / 总正样本数 (Micro Recall)
# 或者 Mean Recall
user_recalls = hits / gt_counts
# 处理分母为0的情况 (没有正样本的用户)
user_recalls = user_recalls.fillna(0)

print(f"\nStacking 在本地验证集上的 Recall@20: {user_recalls.mean():.5f}")

# 对比：如果只用 SLIM 排序会是多少？
recs_slim = df_valid.sort_values(by=[COL_USER, COL_SLIM], ascending=[True, False]).groupby(COL_USER).head(20)
hits_slim = recs_slim[recs_slim[COL_LABEL] == 1].groupby(COL_USER).size().reindex(all_valid_users_idx, fill_value=0)
recall_slim = (hits_slim / gt_counts).fillna(0).mean()
print(f"对比基准 (纯 SLIM) Recall@20: {recall_slim:.5f}")


--- 5. 最终效果评估 ---

[特征重要性排行榜]
           Feature           Gain
3        slim_rank  909181.090940
0       slim_score  314380.809909
6          slim_mm   74471.118984
4        ials_rank   60214.402185
5        user_rank   35487.811262
1       ials_score   34335.025972
9   diff_slim_user   21143.466989
2       user_score   17616.619578
10  diff_slim_ials   16168.273571
7          ials_mm    5898.184387
8          user_mm    4266.373461

Stacking 在本地验证集上的 Recall@20: 0.10892
对比基准 (纯 SLIM) Recall@20: 0.09626


# 2

In [30]:
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.preprocessing import MinMaxScaler
import gc
import os
import scipy.sparse as sp

# --- 配置区域 ---
FOLDS = 5
ALPHA = 0.7  # SLIM 的权重
CANDIDATE_CUTOFF = 200
MODELS_FOLDER_PATH = "k_fold_models"  # 你的模型保存文件夹
OUTPUT_FOLDER = "submission_output"   # 结果输出文件夹
if not os.path.exists(OUTPUT_FOLDER): os.makedirs(OUTPUT_FOLDER)

SUBMISSION_FILENAME = f"submission_bagging_SLIM_IALS_alpha{ALPHA}.csv"
SUBMISSION_PATH = os.path.join(OUTPUT_FOLDER, SUBMISSION_FILENAME)

DATA_TARGET_USERS_PATH = "dataset/data_target_users_test.csv" # 请确认路径

# --- 0. 准备工作 ---
# 加载目标用户
print("正在加载目标用户...")
df_target_users = pd.read_csv(DATA_TARGET_USERS_PATH)
target_user_ids = df_target_users['user_id'].values

# 假设 urm_all 已经在内存中 (包含所有交互的稀疏矩阵)
# 如果没有，你需要先加载它，用于过滤用户已看过的物品
# from data_reader import load_data
# urm_all = load_data(...)

# --- 辅助函数 1: 批量预测 ---
def get_candiates_dataframe(recommender, target_users):
    user_list = []
    item_list = []
    score_list = []

    # 批处理防止内存溢出
    batch_size = 2000

    for start_idx in range(0, len(target_users), batch_size):
        end_idx = min(start_idx + batch_size, len(target_users))
        batch_users = target_users[start_idx:end_idx]

        # 推荐 (自动过滤已读)
        recommendations = recommender.recommend(
            batch_users,
            cutoff=CANDIDATE_CUTOFF,
            remove_seen_flag=True
        )

        # 计算分数
        # 兼容性处理：不同框架 compute_item_score 接口可能不同
        # 这里假设 recommender 能够对 batch users 计算分数
        all_scores_batch = recommender._compute_item_score(batch_users)

        for i, user_id in enumerate(batch_users):
            rec_items = recommendations[i]
            # 提取分数
            rec_scores = all_scores_batch[i, rec_items]

            user_list.extend([user_id] * len(rec_items))
            item_list.extend(rec_items)
            score_list.extend(rec_scores)

    return pd.DataFrame({
        "user_id": user_list,
        "item_id": item_list,
        "score": score_list
    })

# --- 辅助函数 2: Bagging (5-Fold 平均) ---
def predict_bagging_average(recommender_class, model_name_prefix):
    """
    加载 5 个模型，分别预测，然后取分数的平均值
    """
    print(f"\n>>> 开始处理 {model_name_prefix} 的 Bagging (5 Folds)...")

    # 用于存储累加分数
    # 结构: {(user, item): sum_score}
    # 为了效率，我们使用 DataFrame 的 concat + groupby
    all_dfs = []

    for fold_id in range(1, FOLDS + 1): # 假设你保存的是 fold_1 到 fold_5
        # 构造文件名，根据你的保存习惯可能需要微调
        # 之前的代码示例是: f"{ModelName}_fold_{fold_id}"
        file_name = f"{model_name_prefix}_fold_{fold_id}"

        print(f"  [Fold {fold_id}/{FOLDS}] 加载模型: {file_name} ...")

        try:
            # 初始化模型对象 (传入 urm_all 以便全局过滤)
            recommender = recommender_class(urm_all)
            # 加载权重
            recommender.load_model(MODELS_FOLDER_PATH, file_name)

            # 预测
            df_fold = get_candiates_dataframe(recommender, target_user_ids)
            df_fold['fold_id'] = fold_id # 标记一下，虽然后面不用
            all_dfs.append(df_fold)

            # 清理内存
            del recommender
            gc.collect()

        except Exception as e:
            print(f"  !!! 警告: 模型 {file_name} 加载失败或预测出错: {e}")
            print("  将跳过此 Fold。")

    if not all_dfs:
        raise ValueError(f"没有成功加载任何 {model_name_prefix} 模型！")

    print(f"  正在合并 {model_name_prefix} 的 {len(all_dfs)} 个结果...")
    # 合并所有 fold 的结果
    df_concat = pd.concat(all_dfs)

    # 按 (user, item) 分组取平均
    # sum / 5 还是 mean? Mean 更好，因为可能某些 item 只在部分 fold 推荐了
    df_avg = df_concat.groupby(['user_id', 'item_id'])['score'].mean().reset_index()

    return df_avg

# ==========================================
#               主执行流程
# ==========================================

# 1. 计算 SLIM 的 Bagging 平均分
df_slim_avg = predict_bagging_average(SLIMElasticNetRecommender, "SLIM")
df_slim_avg.rename(columns={"score": "slim_score"}, inplace=True)

# 2. 计算 IALS 的 Bagging 平均分
df_ials_avg = predict_bagging_average(IALSRecommender, "IALS")
df_ials_avg.rename(columns={"score": "ials_score"}, inplace=True)

# 3. 合并两张表
print("\n>>> 正在合并 SLIM 和 IALS 结果...")
df_final = pd.merge(df_slim_avg, df_ials_avg, on=["user_id", "item_id"], how="outer")

# 填充缺失值 (没进 Top N 的当作 0 分)
df_final.fillna(0.0, inplace=True)

# 4. 执行 MinMax 归一化 (关键步骤)
# 因为 SLIM 和 IALS 的分数范围不同，直接加权(0.7)会失效，必须归一化
print(">>> 执行 MinMax 归一化...")
scaler = MinMaxScaler()

df_final['slim_mm'] = scaler.fit_transform(df_final['slim_score'].values.reshape(-1, 1))
df_final['ials_mm'] = scaler.fit_transform(df_final['ials_score'].values.reshape(-1, 1))

# 5. 线性加权融合
# Formula: Alpha * SLIM + (1-Alpha) * IALS
print(f">>> 计算最终得分 (Alpha={ALPHA})...")
df_final['final_score'] = ALPHA * df_final['slim_mm'] + (1 - ALPHA) * df_final['ials_mm']

# 6. 排序并生成 Top 20
print(">>> 排序并截取 Top 20...")
submission_df = df_final.sort_values(by=['user_id', 'final_score'], ascending=[True, False])
submission_df = submission_df.groupby('user_id').head(20)

# 7. 写入 CSV
print(f">>> 写入提交文件: {SUBMISSION_PATH}")
grouped_preds = submission_df.groupby('user_id')['item_id'].apply(list)

with open(SUBMISSION_PATH, "w") as f:
    f.write("user_id,item_list\n")
    for user_id in tqdm(target_user_ids):
        # 获取推荐列表
        items = grouped_preds.get(user_id, [])
        # 转字符串
        item_str = " ".join(map(str, items))
        f.write(f"{user_id},{item_str}\n")

print("\n------------------------------------------------------")
print(" 完成！")
print(f" 策略: 5-Fold Bagging + MinMax Scaling")
print(f" 权重: {ALPHA} * SLIM + {1-ALPHA} * IALS")
print(f" 文件已保存至: {SUBMISSION_PATH}")
print("------------------------------------------------------")

正在加载目标用户...

>>> 开始处理 SLIM 的 Bagging (5 Folds)...
  [Fold 1/5] 加载模型: SLIM_fold_1 ...
SLIMElasticNetRecommender: Loading model from file 'k_fold_modelsSLIM_fold_1'
SLIMElasticNetRecommender: Loading complete
  [Fold 2/5] 加载模型: SLIM_fold_2 ...
SLIMElasticNetRecommender: Loading model from file 'k_fold_modelsSLIM_fold_2'
SLIMElasticNetRecommender: Loading complete
  [Fold 3/5] 加载模型: SLIM_fold_3 ...
SLIMElasticNetRecommender: Loading model from file 'k_fold_modelsSLIM_fold_3'
SLIMElasticNetRecommender: Loading complete
  [Fold 4/5] 加载模型: SLIM_fold_4 ...
SLIMElasticNetRecommender: Loading model from file 'k_fold_modelsSLIM_fold_4'
SLIMElasticNetRecommender: Loading complete
  [Fold 5/5] 加载模型: SLIM_fold_5 ...
SLIMElasticNetRecommender: Loading model from file 'k_fold_modelsSLIM_fold_5'
SLIMElasticNetRecommender: Loading complete
  正在合并 SLIM 的 5 个结果...

>>> 开始处理 IALS 的 Bagging (5 Folds)...
  [Fold 1/5] 加载模型: IALS_fold_1 ...
IALSRecommender: Loading model from file 'k_fold_modelsIALS_fold_1'
I

100%|██████████| 27095/27095 [00:00<00:00, 92068.06it/s]


------------------------------------------------------
 完成！
 策略: 5-Fold Bagging + MinMax Scaling
 权重: 0.7 * SLIM + 0.30000000000000004 * IALS
 文件已保存至: submission_output\submission_bagging_SLIM_IALS_alpha0.7.csv
------------------------------------------------------


In [37]:
import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.model_selection import KFold
from sklearn.preprocessing import MinMaxScaler
from tqdm import tqdm
import gc
import os

# --- 配置 ---
FOLDS = 5
RANDOM_SEED = 1234 # 必须与你训练时的 Seed 保持一致！
ALPHA = 0.7      # SLIM 的权重
CANDIDATE_CUTOFF = 200
MODELS_FOLDER_PATH = "k_fold_models"

# 假设 urm_all 是全量交互矩阵
# from data_reader import load_data
# urm_all = load_data(...)
# 这里的 urm_all 必须包含所有用户和物品的交互

# 1. 先将矩阵转换为 COO 格式，这样才有 .row 和 .col 属性
urm_coo = urm_all.tocoo()

# 2. 确保索引是基于这个 COO 矩阵生成的
interactions_indices = np.arange(urm_coo.nnz)

kf = KFold(n_splits=FOLDS, shuffle=True, random_state=RANDOM_SEED)

# --- 辅助函数：快速计算 Recall ---
def calculate_recall(df_preds, urm_val):
    """
    df_preds: 包含 user_id, item_id, final_score (已截断 Top 20)
    urm_val: 验证集的稀疏矩阵 (Ground Truth)
    """
    # 1. 提取 GT (真实交互)
    val_coo = urm_val.tocoo()
    gt_df = pd.DataFrame({
        'user_id': val_coo.row,
        'item_id': val_coo.col
    })

    # 2. 计算每个用户的 GT 数量 (分母)
    gt_counts = gt_df.groupby('user_id').size()

    # 3. 计算命中数 (分子)
    # Inner Join: 只有预测对了的才会留下来
    hits = pd.merge(df_preds, gt_df, on=['user_id', 'item_id'], how='inner')
    hit_counts = hits.groupby('user_id').size()

    # 4. 计算 Recall
    # 对齐索引，填充0
    all_users = np.unique(val_coo.row)
    hit_counts = hit_counts.reindex(all_users, fill_value=0)
    gt_counts = gt_counts.reindex(all_users, fill_value=1) # 防止除以0

    recalls = hit_counts / gt_counts
    return recalls.mean()

# --- 辅助函数：生成候选集 ---
def get_candiates_dataframe(recommender, target_users):
    user_list = []
    item_list = []
    score_list = []

    batch_size = 2000
    for start_idx in range(0, len(target_users), batch_size):
        end_idx = min(start_idx + batch_size, len(target_users))
        batch_users = target_users[start_idx:end_idx]

        recommendations = recommender.recommend(batch_users, cutoff=CANDIDATE_CUTOFF, remove_seen_flag=True)
        all_scores_batch = recommender._compute_item_score(batch_users)

        for i, user_id in enumerate(batch_users):
            rec_items = recommendations[i]
            rec_scores = all_scores_batch[i, rec_items]
            user_list.extend([user_id] * len(rec_items))
            item_list.extend(rec_items)
            score_list.extend(rec_scores)

    return pd.DataFrame({"user_id": user_list, "item_id": item_list, "score": score_list})

# ==========================================
#           开始交叉验证循环
# ==========================================
print(f"--- 开始本地验证 (Alpha={ALPHA}, Folds={FOLDS}) ---")

fold_recalls = []
fold_id = 0

for train_idx, val_idx in kf.split(interactions_indices):
    fold_id += 1
    print(f"\n>>> 正在验证 Fold {fold_id} / {FOLDS} ...")

    # 1. 重建验证集矩阵 (Ground Truth)
    # 3. 在构建矩阵时，使用 urm_coo 而不是 urm_all
    val_mask = np.zeros(urm_coo.nnz, dtype=bool)
    val_mask[val_idx] = True

    URM_val_fold = sp.csr_matrix(
        (urm_coo.data[val_mask], (urm_coo.row[val_mask], urm_coo.col[val_mask])),
        shape=urm_all.shape # shape 可以用原始的
    )

    # 【关键修正】使用正确的 User ID
    target_users_fold = np.unique(urm_coo.row[val_mask])
    print(f"    目标用户数: {len(target_users_fold)}")

    # 2. 加载 SLIM 模型
    # 注意：这里需要重新初始化一个全新的对象，避免内存残留
    rec_slim = SLIMElasticNetRecommender(urm_all)
    rec_slim.load_model(MODELS_FOLDER_PATH, f"SLIM_fold_{fold_id}")

    print("    生成 SLIM 预测...")
    df_slim = get_candiates_dataframe(rec_slim, target_users_fold)
    df_slim.rename(columns={"score": "slim_score"}, inplace=True)
    del rec_slim
    gc.collect()

    # 3. 加载 IALS 模型
    rec_ials = IALSRecommender(urm_all)
    rec_ials.load_model(MODELS_FOLDER_PATH, f"IALS_fold_{fold_id}")

    print("    生成 IALS 预测...")
    df_ials = get_candiates_dataframe(rec_ials, target_users_fold)
    df_ials.rename(columns={"score": "ials_score"}, inplace=True)
    del rec_ials
    gc.collect()

    # 4. 合并 & 融合
    df_fold = pd.merge(df_slim, df_ials, on=["user_id", "item_id"], how="outer")
    df_fold.fillna(0.0, inplace=True)

    # 5. MinMax 归一化 (Fold 内部归一化)
    scaler = MinMaxScaler()
    df_fold['slim_mm'] = scaler.fit_transform(df_fold['slim_score'].values.reshape(-1, 1))
    df_fold['ials_mm'] = scaler.fit_transform(df_fold['ials_score'].values.reshape(-1, 1))

    # 6. 加权得分
    df_fold['final_score'] = ALPHA * df_fold['slim_mm'] + (1 - ALPHA) * df_fold['ials_mm']

    # 7. 排序取 Top 20
    df_top20 = df_fold.sort_values(by=['user_id', 'final_score'], ascending=[True, False])
    df_top20 = df_top20.groupby('user_id').head(20)

    # 8. 计算本折 Recall
    recall = calculate_recall(df_top20, URM_val_fold)
    fold_recalls.append(recall)
    print(f"    Fold {fold_id} Recall@20: {recall:.5f}")

# ==========================================
#           最终结果
# ==========================================
print("\n================ 验证结果 ================")
print(f"各折分数: {fold_recalls}")
print(f"本地平均 Recall@20: {np.mean(fold_recalls):.5f}")
print("==========================================")

--- 开始本地验证 (Alpha=0.7, Folds=5) ---

>>> 正在验证 Fold 1 / 5 ...
    目标用户数: 27059
SLIMElasticNetRecommender: Loading model from file 'k_fold_modelsSLIM_fold_1'
SLIMElasticNetRecommender: Loading complete
    生成 SLIM 预测...
IALSRecommender: Loading model from file 'k_fold_modelsIALS_fold_1'
IALSRecommender: Loading complete
    生成 IALS 预测...
    Fold 1 Recall@20: 0.00000

>>> 正在验证 Fold 2 / 5 ...
    目标用户数: 27062
SLIMElasticNetRecommender: Loading model from file 'k_fold_modelsSLIM_fold_2'
SLIMElasticNetRecommender: Loading complete
    生成 SLIM 预测...
IALSRecommender: Loading model from file 'k_fold_modelsIALS_fold_2'
IALSRecommender: Loading complete
    生成 IALS 预测...
    Fold 2 Recall@20: 0.00000

>>> 正在验证 Fold 3 / 5 ...
    目标用户数: 27059
SLIMElasticNetRecommender: Loading model from file 'k_fold_modelsSLIM_fold_3'
SLIMElasticNetRecommender: Loading complete
    生成 SLIM 预测...
IALSRecommender: Loading model from file 'k_fold_modelsIALS_fold_3'
IALSRecommender: Loading complete
    生成 IALS 预测..

KeyboardInterrupt: 